## Lesson 4: Solution Provisioning - The "Root Cause" vs. "Symptom"
You’ve built the code and optimized the query. Now, we move to the Thinking Process.

A 1x Analyst fixes the Symptom (The code was used too much).
A 100x Analyst fixes the Root Cause (Why was the code even able to be used like this?).

#### The Scenario
You find that the "SPRING20" abuse happened because the web-front-end allowed "Guest Checkout" users to apply the code without validating their email address. People were simply refreshing their browser and using it again.

#### Your Solution Provisioning Task
Instead of just saying "I'll keep the script running," what is the long-term engineering solution you would propose to the dev team?

#### Think about:
- Validation: Where should it happen?
- State: How do we track a user if they aren't logged in?

~~Can that really be the issue? if they are users, then definitely they have completed registration. Also this does not account for the stacking of code~~

#### 1. The "User" Illusion (Identity vs. Account)
"If they are users, then definitely they have completed registration."

In a perfect world, yes. But in a 100x world, you must account for Guest Checkout.

Many e-commerce sites allow you to buy without a password.

The system creates a "Shadow User ID" (e.g., user_999) in the database linked to an email address, but the person hasn't "registered."

**The Exploit:** A bot or a savvy abuser uses email+1@gmail.com, email+2@gmail.com, etc. To the database, these are 100 different users, but to the business, it's one person stealing 100 discounts.

100x Solution: We don't just track user_id. We track Fingerprints:
- IP Address
- Device ID

Payment Method (If 10 different "users" use the same Credit Card, that’s an immediate fraud flag).

#### 2. The "Stacking" Problem (Logic Collision)
You are absolutely right—my previous SQL didn't fully solve for Stacking (using two different codes on one order).

If a user applies SPRING20 (20% off) AND WELCOME10 (10% off), and the system allows it, they get 30% off. This is a Logic Collision.

The 100x SQL to Catch Stacking:
We need to shift the grain from "How many times was this code used?" to "How many codes were used on this specific Order?"

In [ ]:
SELECT 
    order_id,
    user_id,
    COUNT(DISTINCT discount_code) as unique_codes_applied,
    GROUP_CONCAT(discount_code) as codes_used, -- See exactly which ones were stacked
    SUM(discount_amount) as total_saved,
    order_total_before_discount
FROM order_discounts
GROUP BY 1, 2
HAVING COUNT(DISTINCT discount_code) > 1 -- This catches the "Stackers"
   OR SUM(discount_amount) > (order_total_before_discount * 0.25); -- Catch anything over a 25% cap

#### The Engineering Fix (Root Cause)
Since you’ve identified that "Registration" and "Stacking" are the real issues, here is the Solution Provision you propose to the Engineering/Product Team:

#### The 3-Tier Defense Strategy:
- The API Layer (Prevention): The backend must check the order_discounts table before the order is finalized. If COUNT(order_id) > 0 for that session, reject the second code.
- The Identity Layer (Fingerprinting): Implement a check that blocks more than 3 guest checkouts from the same IP address/Device ID within 24 hours.
- The Database Layer (Constraints): Add a UNIQUE constraint or a "Trigger" that prevents more than one row entry per order_id in the order_discounts table.

#### Your Final 30-Minute Challenge for Today
Since we’ve covered the code, the logic, and the communication—let’s talk Data Quality.

If you run your 'Stacking' query and find 1,000 orders with multiple codes, but the 'Order Total' is still positive, how do you determine which orders are 'Fraud' and which are just 'Lucky Customers' who found a bug? How do you prioritize which ones to investigate first? (Hint: Think about Magnitude vs. Frequency).

## Wrap Up!
To wrap up the Fraud Priority logic, we need to move from "finding everything" to "fixing what matters." A 100x Analyst uses a Risk Matrix to decide where to spend their 30 minutes.

#### 1. The Logic: Magnitude vs. Frequency
If you have 1,000 orders, you don't look at them alphabetically. You plot them on two axes:
- Magnitude (The "Bleed"): How much money did we lose per order? (e.g., a $200 discount on a $210 order).
- Frequency (The "Pattern"): How many times did this specific user or IP address do it?- 

**The Prioritization Rule:**
- High Frequency + High Magnitude = Professional Fraud (Immediate Block).
- Low Frequency + High Magnitude = System Bug/Lucky User (Fix the code, maybe don't penalize).

#### 2. The SQL Solution (The "Priority Score")
We can actually calculate a "Suspect Score" directly in SQL to rank our investigation list.



In [ ]:
SELECT 
    user_id,
    COUNT(order_id) as total_orders,
    AVG(discount_amount / order_total_before_discount) as avg_discount_depth,
    SUM(discount_amount) as total_revenue_loss,
    -- 100x Logic: Create a weighted score
    (COUNT(order_id) * 0.7) + (SUM(discount_amount) * 0.3) as risk_score
FROM order_discounts
GROUP BY 1
HAVING total_revenue_loss > 0
ORDER BY risk_score DESC
LIMIT 50;